[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/12_montecarlo/notebooks/aplicaciones/03_opcion_financiera.ipynb)

# Aplicación 3: Pricing de Opciones Financieras

## El problema

Una **opción de compra europea** (European call option) es un contrato financiero que da a su dueño el derecho (pero no la obligación) de **comprar una acción** a precio $K$ (el *strike price*) en una fecha futura $T$.

Si el precio de la acción en $T$ es $S_T$:
- Si $S_T > K$: el dueño ejerce la opción, gana $S_T - K$ (compra barato, vende al precio de mercado)
- Si $S_T \leq K$: el dueño no ejerce la opción, pierde 0

El **payoff** es:
$$\text{Payoff} = \max(S_T - K, 0) = (S_T - K)^+$$

### ¿Cuánto vale la opción hoy?

Por el principio de no-arbitraje, el precio justo de la opción hoy es el **valor esperado del payoff, descontado** a la tasa libre de riesgo $r$:

$$\text{Precio} = e^{-rT} \cdot \mathbb{E}[\max(S_T - K, 0)]$$

Para calcular esta esperanza necesitamos un modelo para $S_T$. El modelo estándar es el **Movimiento Browniano Geométrico (GBM)**.

In [ ]:
# Descomenta si estás en Colab
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(42)
print("Listo ✓")

---
## Parte 1: El Modelo GBM

El **Movimiento Browniano Geométrico** modela el precio de una acción con la SDE:

$$dS = \mu S\, dt + \sigma S\, dW_t$$

donde:
- $\mu$: drift (tendencia de crecimiento esperado)
- $\sigma$: volatilidad
- $dW_t$: incremento de Movimiento Browniano ($\sim \mathcal{N}(0, dt)$)

La solución exacta es:

$$S_t = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W_t\right]$$

Para simular: dividimos $[0, T]$ en $M$ pasos de tamaño $\Delta t = T/M$:

$$S_{t+\Delta t} = S_t \cdot \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\, Z\right], \qquad Z \sim \mathcal{N}(0,1)$$

**Nota**: en pricing de opciones, el modelo se usa bajo la **medida neutral al riesgo** donde el drift es la tasa libre de riesgo $r$ (no $\mu$). Este es un resultado del teorema de Girsanov, y es lo que permite el pricing por no-arbitraje.

In [ ]:
# Market parameters
S0 = 100.0      # current stock price
K = 105.0       # strike price
T = 1.0         # time to expiry (years)
r = 0.05        # risk-free rate
sigma = 0.20    # volatility
M = 252         # number of time steps (trading days per year)

def simulate_gbm(S0, r, sigma, T, M, N, seed=None):
    """Simulate N paths of GBM with M time steps.
    Returns array of shape (N, M+1).
    """
    rng_local = np.random.default_rng(seed)
    dt = T / M
    Z = rng_local.standard_normal((N, M))
    log_increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    S = np.zeros((N, M + 1))
    S[:, 0] = S0
    for t in range(M):
        S[:, t+1] = S[:, t] * np.exp(log_increments[:, t])
    return S

# Simulate and visualize 20 paths
N_vis = 20
paths = simulate_gbm(S0, r, sigma, T, M, N_vis, seed=42)
t_grid = np.linspace(0, T, M + 1)

fig, ax = plt.subplots(figsize=(11, 5))
for i in range(N_vis):
    color = "#E94F37" if paths[i, -1] > K else "#2E86AB"
    ax.plot(t_grid, paths[i], lw=0.8, alpha=0.7, color=color)
ax.axhline(K, color="black", ls="--", lw=1.5, label=f"Strike $K={K}$")
ax.axhline(S0, color="gray", ls=":", lw=1, label=f"$S_0={S0}$")
ax.set_xlabel("Tiempo (años)")
ax.set_ylabel("Precio $S_t$")
ax.set_title("Caminos GBM — rojo: $S_T > K$ (opción gana) · azul: $S_T \\leq K$ (opción expira sin valor)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Parte 2: Pricing Monte Carlo de la Opción Europea

Para la opción europea, solo nos importa el precio **terminal** $S_T$ (no el camino completo).

La fórmula exacta de Black-Scholes nos da el precio teórico:

$$C_{\text{BS}} = S_0 \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

donde $d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$, $\Phi$ = CDF normal.

Usaremos esto como **ground truth** para validar nuestro estimador MC.

In [ ]:
def black_scholes_call(S0, K, T, r, sigma):
    """Black-Scholes European call option price."""
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

bs_price = black_scholes_call(S0, K, T, r, sigma)
print(f"Precio Black-Scholes: {bs_price:.4f}")

def mc_european_call(S0, K, T, r, sigma, N, seed=None):
    """Monte Carlo price of European call option."""
    rng_local = np.random.default_rng(seed)
    # Only need terminal price: S_T = S_0 * exp((r - sigma^2/2)*T + sigma*sqrt(T)*Z)
    Z = rng_local.standard_normal(N)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.maximum(S_T - K, 0)
    price_hat = np.exp(-r*T) * payoffs.mean()
    se = np.exp(-r*T) * payoffs.std(ddof=1) / np.sqrt(N)
    return price_hat, se

print("\nEstimados MC:")
print(f"{'N':>10} | {'Precio MC':>12} | {'SE':>10} | {'IC 95%':>20} | {'Error vs BS':>12}")
print("-" * 75)
for N in [1_000, 10_000, 100_000, 1_000_000]:
    price, se = mc_european_call(S0, K, T, r, sigma, N, seed=42)
    lo, hi = price - 1.96*se, price + 1.96*se
    print(f"{N:>10,} | {price:>12.4f} | {se:>10.4f} | [{lo:.4f}, {hi:.4f}] | {abs(price-bs_price):>12.4f}")

### 🔧 Ejercicio 1: Impacto de la volatilidad

Experimenta con diferentes valores de $\sigma$:

1. Estima el precio de la opción para $\sigma \in \{0.1, 0.2, 0.3, 0.4, 0.5\}$
2. ¿El precio sube o baja cuando $\sigma$ aumenta? ¿Por qué intuitivamente?
3. Grafica precio vs. $\sigma$ para MC y Black-Scholes (deben coincidir)

In [ ]:
sigmas = np.linspace(0.05, 0.6, 20)  # <-- CHANGE THIS
N = 100_000

bs_prices = [black_scholes_call(S0, K, T, r, s) for s in sigmas]
mc_prices = [mc_european_call(S0, K, T, r, s, N, seed=42)[0] for s in sigmas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sigmas, bs_prices, "-", color="#E94F37", lw=2, label="Black-Scholes (exacto)")
ax.plot(sigmas, mc_prices, "o", color="#2E86AB", ms=5, alpha=0.8, label=f"Monte Carlo ($N={N:,}$)")
ax.set_xlabel("Volatilidad $\\sigma$")
ax.set_ylabel("Precio de la opción")
ax.set_title("Precio de opción europea vs. volatilidad")
ax.legend()
plt.tight_layout()
plt.show()
print("¿Qué observas? ¿Mayor volatilidad = mayor o menor precio?")

---
## Parte 3: Opción Asiática — cuando Black-Scholes no aplica

Una **opción asiática** tiene un payoff que depende del **promedio del precio** durante el período:

$$\text{Payoff}_{\text{asiática}} = \max\left(\bar{S} - K, 0\right), \qquad \bar{S} = \frac{1}{M}\sum_{t=1}^M S_{t\Delta t}$$

Para este tipo de opciones, **no existe fórmula cerrada exacta** en el modelo GBM. Black-Scholes no aplica. Monte Carlo es la herramienta estándar en la industria.

Esto es un ejemplo perfecto de cuándo MC es indispensable.

In [ ]:
def mc_asian_call(S0, K, T, r, sigma, M, N, seed=None):
    """Monte Carlo price of Asian (arithmetic average) call option."""
    paths = simulate_gbm(S0, r, sigma, T, M, N, seed=seed)
    # Average price along each path (excluding t=0)
    S_avg = paths[:, 1:].mean(axis=1)
    payoffs = np.maximum(S_avg - K, 0)
    price_hat = np.exp(-r*T) * payoffs.mean()
    se = np.exp(-r*T) * payoffs.std(ddof=1) / np.sqrt(N)
    return price_hat, se

N = 50_000
asian_price, asian_se = mc_asian_call(S0, K, T, r, sigma, M, N, seed=42)
euro_price, euro_se = mc_european_call(S0, K, T, r, sigma, N, seed=42)

print(f"Opción Europea (Black-Scholes exacto): {bs_price:.4f}")
print(f"Opción Europea (MC, N={N:,}):   {euro_price:.4f} ± {1.96*euro_se:.4f}")
print(f"Opción Asiática (MC, N={N:,}):  {asian_price:.4f} ± {1.96*asian_se:.4f}")
print(f"\nLa opción asiática es más barata: el promedio es menos volátil que el precio final.")

---
## Parte 4: Reducción de Varianza en Pricing

Para la opción europea, podemos aplicar **antithetic variates**: si $Z$ genera un camino $S_T$, entonces $-Z$ genera el camino "reflejo".

La clave: si $S_T^+$ (del camino con $Z$) es alto, $S_T^-$ (del camino con $-Z$) es bajo, y viceversa — correlación negativa perfecta en las Gaussianas, lo que traduce a correlación negativa en los payoffs (si $f$ es monótona en $S_T$, que lo es).

In [ ]:
def mc_european_antithetic(S0, K, T, r, sigma, N, seed=None):
    """Antithetic variates MC for European call."""
    rng_local = np.random.default_rng(seed)
    Z = rng_local.standard_normal(N // 2)
    # Two paths: Z and -Z
    S_T_plus  = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    S_T_minus = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-Z))
    payoffs_plus  = np.maximum(S_T_plus - K, 0)
    payoffs_minus = np.maximum(S_T_minus - K, 0)
    paired_payoffs = (payoffs_plus + payoffs_minus) / 2
    price_hat = np.exp(-r*T) * paired_payoffs.mean()
    se = np.exp(-r*T) * paired_payoffs.std(ddof=1) / np.sqrt(N // 2)
    return price_hat, se

n_reps = 500
Ns = [1_000, 5_000, 20_000]

print(f"{'N':>10} | {'MC Crudo SE':>14} | {'Antithetic SE':>14} | {'Reducción SE':>14}")
print("-" * 60)
for N in Ns:
    crude_ses, anti_ses = [], []
    for i in range(n_reps):
        _, se_c = mc_european_call(S0, K, T, r, sigma, N, seed=i)
        _, se_a = mc_european_antithetic(S0, K, T, r, sigma, N, seed=i)
        crude_ses.append(se_c)
        anti_ses.append(se_a)
    se_c_mean = np.mean(crude_ses)
    se_a_mean = np.mean(anti_ses)
    print(f"{N:>10,} | {se_c_mean:>14.5f} | {se_a_mean:>14.5f} | {(1-se_a_mean/se_c_mean)*100:>13.1f}%")

---
## Parte 5: 🔧 Ejercicio — Opción Barrera

Una **opción barrera knock-out** se comporta como una opción europea pero **se anula automáticamente** si el precio de la acción cruza un nivel de barrera $B$ en cualquier momento durante el período.

$$\text{Payoff} = \max(S_T - K, 0) \cdot \mathbf{1}\left[\max_{0 \leq t \leq T} S_t < B\right]$$

Para esta opción:
1. **¿Puedes calcularla analíticamente?** Black-Scholes estándar no aplica directamente (aunque sí existe una fórmula cerrada para el caso continuo — investiga)
2. **Implementa el pricing con MC** usando los caminos discretos
3. **Experimenta**: ¿cómo cambia el precio cuando $B$ sube de 110 a 150 a 200?
4. **Bias de discretización**: con pasos discretos, ¿es exacto el precio MC? ¿Qué pasa si aumentas $M$?

In [ ]:
def mc_barrier_call(S0, K, B, T, r, sigma, M, N, seed=None):
    """MC price of knock-out barrier European call."""
    paths = simulate_gbm(S0, r, sigma, T, M, N, seed=seed)
    # Option is alive only if max(S_t) < B
    max_price = paths.max(axis=1)
    alive = max_price < B
    S_T = paths[:, -1]
    payoffs = np.maximum(S_T - K, 0) * alive
    price_hat = np.exp(-r*T) * payoffs.mean()
    se = np.exp(-r*T) * payoffs.std(ddof=1) / np.sqrt(N)
    return price_hat, se, alive.mean()

N = 100_000
barriers = [110, 120, 130, 150, 200, 1000]  # <-- CHANGE THIS

print(f"{'Barrera B':>12} | {'Precio':>10} | {'SE':>8} | {'Prob. viva':>12}")
print("-" * 50)
for B in barriers:
    price, se, prob_alive = mc_barrier_call(S0, K, B, T, r, sigma, M, N, seed=42)
    print(f"{B:>12} | {price:>10.4f} | {se:>8.4f} | {prob_alive:>12.2%}")

print(f"\nOpción europea (sin barrera): {bs_price:.4f}")
print("(B=1000 ≈ sin barrera, debe coincidir con Black-Scholes)")

---
## Reflexión

| Opción | ¿Fórmula cerrada? | ¿MC funciona? | Notas |
|--------|:-----------------:|:-------------:|-------|
| Europea (call/put) | ✓ Black-Scholes | ✓ | MC valida la fórmula exacta |
| Asiática (promedio) | ✗ (aprox. disponibles) | ✓ | MC es el método estándar |
| Barrera | ✓ (caso continuo) | ✓ (con bias de discretización) | Cuidado con el número de pasos M |
| Americana (ejercicio anticipado) | ✗ | ✓ (con Longstaff-Schwartz) | MC requiere variante especial |

**La moraleja**: Monte Carlo es la herramienta estándar en la industria financiera precisamente porque escala bien a opciones complejas donde no hay fórmulas cerradas. El mismo motor (simular caminos del GBM) sirve para cualquier payoff.

---
**Recursos adicionales:**
- *Options, Futures, and Other Derivatives* — Hull (el libro estándar de la industria)
- El paper original de Black y Scholes (1973) es sorprendentemente legible